# Tarea 4 (versión simplificada): Clasificación de Crisis con Modelos Multimodales
**Versión reducida — pensada para completarse en ~1 hora.**

En esta tarea utilizaremos el *Multimodal Crisis Dataset* (https://crisisnlp.qcri.org/crisismmd) para clasificar tweets de desastres naturales combinando **texto** e **imagen**.

La mayor parte de la infraestructura (carga de datos, encoders preentrenados, `Dataset`, bucle de entrenamiento) ya está implementada. Tú solo debes completar:
1. Los **módulos de fusión** (Naive Fusion y Cross-Modal Attention).
2. El **lanzamiento de los 3 entrenamientos** (instanciar el modelo y llamar a `train_model`).

Al final, además probaremos un modelo **instruido (zero-shot)**, `Qwen3-VL-2B-Instruct`, sin ningún entrenamiento.

## El dataset

Cada ejemplo incluye un **tweet** y su **imagen correspondiente**, junto con una **etiqueta** que pertenece al conjunto:

- `Affected individuals`
- `Infrastructure and utility damage`
- `Not humanitarian`
- `Other relevant information`
- `Rescue volunteering or donation effort`

Para que el laboratorio sea rápido, trabajaremos con una **muestra reducida** del dataset (ver `N_TRAIN`, `N_DEV`, `N_TEST` más abajo).

In [ ]:
!wget https://users.dcc.uchile.cl/~vbarrier/ECI2026_MModal_GenAI/data/CrisisMMD_v2.0.tar.gz
# descomprimir el dataset
!tar zxvf CrisisMMD_v2.0.tar.gz

In [ ]:
!wget https://users.dcc.uchile.cl/~vbarrier/ECI2026_MModal_GenAI/data/crisismmd_datasplit_agreed_label.zip
!unzip -o crisismmd_datasplit_agreed_label.zip

In [ ]:
## Carga de datos (dado) ##
import pandas as pd
import os
import numpy as np

def load_crisismmd_tsv(path):
    '''Load tsv file that contains annotations for task in CrisisMMD dataset'''
    rows = []
    with open(path, encoding="utf-8") as f:
        header = f.readline().strip().split("\t")
        expected_cols = len(header)

        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) == expected_cols:
                rows.append(parts)
            elif len(parts) > expected_cols:
                fixed = parts[:4] + [" ".join(parts[4:len(parts)-(expected_cols-5)])] + parts[-(expected_cols-5):]
                rows.append(fixed)
            else:
                print("skiping line")
                continue

    df = pd.DataFrame(rows, columns=header)
    return df

def load_annotations(task="humanitarian", data_dir="/content/"):
    '''Load annotations for all splits'''
    datasplits = {}
    for split in ["train", "dev", "test"]:
        df = load_crisismmd_tsv(os.path.join(data_dir, f"crisismmd_datasplit_agreed_label/task_{task}_text_img_agreed_lab_{split}.tsv"))
        df = df.rename(columns={"image": "image_path"})
        if split == "dev":  # val es lo mismo que dev
            datasplits["val"] = df
        else:
            datasplits[split] = df
    return datasplits

def read_data(annotations, split, n_samples=None, seed=42):
    '''Lee un split y opcionalmente toma una submuestra de n_samples ejemplos'''
    df = annotations[split][['tweet_text', 'label_text', 'image_path']].reset_index(drop=True)
    if n_samples is not None and n_samples < len(df):
        df = df.sample(n=n_samples, random_state=seed).reset_index(drop=True)
    X = df['tweet_text']
    y = df['label_text']
    path_image = df['image_path']
    return X, y, path_image

annotations = load_annotations(task="humanitarian", data_dir="")

In [ ]:
# Submuestra pequeña del dataset para que el lab dure <1 hora
N_TRAIN, N_DEV, N_TEST = 1500, 250, 250
use_subset = False

if use_subset: 
    X_train, y_train, path_image_train = read_data(annotations, "train", n_samples=N_TRAIN)
    X_dev, y_dev, path_image_dev       = read_data(annotations, "val", n_samples=N_DEV)
    X_test, y_test, path_image_test    = read_data(annotations, "test", n_samples=N_TEST)
else:
    X_train, y_train, path_image_train = read_data(annotations, "train")
    X_dev, y_dev, path_image_dev       = read_data(annotations, "val")
    X_test, y_test, path_image_test    = read_data(annotations, "test") 

print(f"Train: {len(X_train)} | Dev: {len(X_dev)} | Test: {len(X_test)}")

In [ ]:
# Ruta a la carpeta de imágenes tras descomprimir el dataset
image_folder = "./CrisisMMD_v2.0/"

path_image_train = [os.path.join(image_folder, p) for p in path_image_train]
path_image_dev   = [os.path.join(image_folder, p) for p in path_image_dev]
path_image_test  = [os.path.join(image_folder, p) for p in path_image_test]

## Exploremos el dataset

Antes de entrenar cualquier modelo, siempre es buena práctica mirar los datos: ¿cómo se distribuyen las clases? ¿cómo se ven los tweets e imágenes?

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

y_train.value_counts().plot(kind="bar")
plt.title("Distribución de clases (train)")
plt.ylabel("Cantidad de ejemplos")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Mostramos un ejemplo de cada clase: imagen + tweet + etiqueta
classes_sorted = sorted(y_train.unique())
sample_idx = [y_train[y_train == c].index[0] for c in classes_sorted]

fig, axes = plt.subplots(1, len(sample_idx), figsize=(4 * len(sample_idx), 4))
for ax, idx in zip(axes, sample_idx):
    img = Image.open(path_image_train[idx]).convert("RGB")
    ax.imshow(img)
    ax.set_title(y_train.loc[idx], fontsize=9)
    ax.axis("off")
plt.suptitle("Un ejemplo por clase del dataset CrisisMMD (imagen + etiqueta)")
plt.tight_layout()
plt.show()

for idx in sample_idx:
    print(f"Tweet: {X_train.loc[idx]}")
    print(f"Etiqueta: {y_train.loc[idx]}")
    print("-" * 80)

## Preparación: encoders de texto e imagen (dado)

Usaremos:
* **RoBERTa** (`roberta-base`) como encoder de texto — tomamos el token `<s>` (equivalente al `[CLS]`) de `last_hidden_state` como representación del tweet.
* **EfficientNet-B3** preentrenado en ImageNet como encoder de imagen — le quitamos el clasificador para quedarnos con el embedding (1536-d).

Definimos una función `build_encoders()` que crea una **copia nueva** de ambos encoders cada vez que la llamamos, para que los 3 experimentos no compartan (ni contaminen) los mismos pesos.

In [ ]:
import torch
from torch import nn
from torch.nn import MultiheadAttention
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModel, AutoTokenizer
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TEXT_EMB_DIM = 768    # hidden size de roberta-base
IMAGE_EMB_DIM = 1536  # dimensión de salida de EfficientNet-B3 sin clasificador

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def build_encoders():
    """Crea instancias frescas de los encoders preentrenados de texto e imagen."""
    text_encoder = AutoModel.from_pretrained("roberta-base")
    image_encoder = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
    image_encoder.classifier = nn.Identity()  # nos quedamos con el embedding (1536-d)
    return text_encoder, image_encoder

In [ ]:
# Transformaciones de imagen: con data augmentation para train, sin para dev/test
image_transform_train = transforms.Compose([
    transforms.RandomResizedCrop(300),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=0.2 * 360),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

image_transform_eval = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
class MultimodalDataset(Dataset):
    """Dataset que retorna (input_ids, attention_mask, imagen, etiqueta) para cada ejemplo."""

    def __init__(self, texts, image_paths, labels, tokenizer, image_transform, max_length=64):
        self.texts = list(texts)
        self.image_paths = list(image_paths)
        self.labels = labels
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        image = Image.open(self.image_paths[idx]).convert("RGB")
        image = self.image_transform(image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_ids, attention_mask, image, label

In [ ]:
# Codificamos las etiquetas como enteros
le = LabelEncoder()
le.fit(list(y_train) + list(y_dev) + list(y_test))
y_train_enc = le.transform(y_train)
y_dev_enc   = le.transform(y_dev)
y_test_enc  = le.transform(y_test)
num_classes = len(le.classes_)
print("Clases:", list(le.classes_))

train_dataset = MultimodalDataset(X_train, path_image_train, y_train_enc, tokenizer, image_transform_train)
dev_dataset   = MultimodalDataset(X_dev, path_image_dev, y_dev_enc, tokenizer, image_transform_eval)
test_dataset  = MultimodalDataset(X_test, path_image_test, y_test_enc, tokenizer, image_transform_eval)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Entrenamiento (dado)

La función `train_model` entrena y evalúa un modelo por `num_epochs` épocas y retorna las curvas de loss/accuracy (train y dev). Está **completamente implementada** — no necesitas modificarla, solo usarla más abajo para lanzar cada experimento.

In [ ]:
def train_model(model, train_loader, dev_loader, num_epochs=3, lr=1e-4):
    """Bucle de entrenamiento genérico (dado): retorna las curvas de loss/accuracy."""
    model.to(device)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, dev_losses = [], []
    train_accs, dev_accs = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss, all_preds, all_labels = 0.0, [], []
        for input_ids, attention_mask, image, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [train]"):
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            image, label = image.to(device), label.to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask, image)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(label.cpu().numpy())

        train_losses.append(running_loss / len(train_loader))
        train_accs.append(accuracy_score(all_labels, all_preds))

        model.eval()
        running_loss, all_preds, all_labels = 0.0, [], []
        with torch.no_grad():
            for input_ids, attention_mask, image, label in tqdm(dev_loader, desc=f"Epoch {epoch+1}/{num_epochs} [dev]"):
                input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
                image, label = image.to(device), label.to(device)

                logits = model(input_ids, attention_mask, image)
                loss = criterion(logits, label)

                running_loss += loss.item()
                all_preds.extend(logits.argmax(dim=1).cpu().numpy())
                all_labels.extend(label.cpu().numpy())

        dev_loss = running_loss / len(dev_loader)
        dev_acc = accuracy_score(all_labels, all_preds)
        dev_f1 = f1_score(all_labels, all_preds, average="macro")
        dev_losses.append(dev_loss)
        dev_accs.append(dev_acc)

        print(f"Epoch {epoch+1}: train_loss={train_losses[-1]:.3f} train_acc={train_accs[-1]:.3f} | "
              f"dev_loss={dev_loss:.3f} dev_acc={dev_acc:.3f} dev_f1={dev_f1:.3f}")

    return train_losses, dev_losses, train_accs, dev_accs


def print_plots_losses(train_losses, dev_losses):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Training Loss")
    plt.plot(dev_losses, label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss Curve")
    plt.legend()
    plt.show()


def evaluate_model(model, loader):
    """Evalúa un modelo ya entrenado sobre un DataLoader (dado): retorna (accuracy, f1_macro)."""
    model.to(device)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, image, label in loader:
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            image, label = image.to(device), label.to(device)

            logits = model(input_ids, attention_mask, image)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, f1

## 📝 Ejercicio 1: Fusión Naive (COMPLETAR)

Completa el método `forward` de `NaiveFusionClassifier`:
1. Obtén el embedding de texto con `self.text_encoder(...)` y toma el token `<s>` (posición 0 de `last_hidden_state`).
2. Obtén el embedding de imagen con `self.image_encoder(image)`.
3. Concatena ambos embeddings (`dim=1`).
4. Pasa el resultado por `self.classifier` y retorna los logits.

In [ ]:
class NaiveFusionClassifier(nn.Module):
    def __init__(self, text_encoder, image_encoder, num_classes, freeze_text=True, freeze_image=True,
                 text_dim=TEXT_EMB_DIM, image_dim=IMAGE_EMB_DIM, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.text_encoder = text_encoder
        self.image_encoder = image_encoder

        for param in self.text_encoder.parameters():
            param.requires_grad = not freeze_text
        for param in self.image_encoder.parameters():
            param.requires_grad = not freeze_image

        self.classifier = nn.Sequential(
            nn.Linear(text_dim + image_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, attention_mask, image):
        # TODO: implementa la fusión naive (concatenación) de texto e imagen
        # 1. text_embedding = ...  (token <s> de self.text_encoder(...).last_hidden_state)
        # 2. image_embedding = ...  (self.image_encoder(image))
        # 3. combined = torch.cat((text_embedding, image_embedding), dim=1)
        # 4. return self.classifier(combined)
        raise NotImplementedError("Completa el forward de NaiveFusionClassifier")

## 📝 Ejecuta los Experimentos 1 y 2 (Naive Fusion)

Ahora que completaste `NaiveFusionClassifier`, lánzalo dos veces con distintas configuraciones. Para cada experimento: (a) crea encoders frescos con `build_encoders()`, (b) instancia `NaiveFusionClassifier`, (c) llama a `train_model(...)` (usa `num_epochs=3`), y (d) guarda los históricos con los nombres de variable indicados (los necesitaremos para comparar y para el gráfico final).

* **Experimento 1 — Naive Fusion, todo congelado:** `NaiveFusionClassifier(..., freeze_text=True, freeze_image=True)` → guarda en `exp1_train_losses, exp1_dev_losses, exp1_train_accs, exp1_dev_accs`.
* **Experimento 2 — Naive Fusion, todo entrenable:** `NaiveFusionClassifier(..., freeze_text=False, freeze_image=False)` → guarda en `exp2_train_losses, exp2_dev_losses, exp2_train_accs, exp2_dev_accs`.

In [ ]:
# TODO: Experimento 1 — Naive Fusion con RoBERTa y EfficientNet CONGELADOS (frozen)
# text_encoder, image_encoder = build_encoders()
# model_exp1 = NaiveFusionClassifier(text_encoder, image_encoder, num_classes=num_classes,
#                                     freeze_text=True, freeze_image=True)
# exp1_train_losses, exp1_dev_losses, exp1_train_accs, exp1_dev_accs = train_model(
#     model_exp1, train_loader, dev_loader, num_epochs=3
# )
# print_plots_losses(exp1_train_losses, exp1_dev_losses)

In [ ]:
# TODO: Experimento 2 — Naive Fusion con RoBERTa y EfficientNet TODO ENTRENABLE
# text_encoder, image_encoder = build_encoders()
# model_exp2 = NaiveFusionClassifier(text_encoder, image_encoder, num_classes=num_classes,
#                                     freeze_text=False, freeze_image=False)
# exp2_train_losses, exp2_dev_losses, exp2_train_accs, exp2_dev_accs = train_model(
#     model_exp2, train_loader, dev_loader, num_epochs=3, lr=1e-5
# )
# print_plots_losses(exp2_train_losses, exp2_dev_losses)

## 📝 Ejercicio 2: Fusión Cross-Modal (COMPLETAR)

Completa el método `forward` de `CrossModalFusionClassifier`:
1. Calcula el embedding de texto (token `<s>`) y el embedding de imagen, igual que en `NaiveFusionClassifier`.
2. Proyecta ambos con `self.text_proj` / `self.image_proj` a `hidden_dim`, y agrega una dimensión de secuencia con `.unsqueeze(1)` (`MultiheadAttention` espera `[batch, seq_len, dim]`).
3. Calcula la atención cruzada texto→imagen: `self.text_to_image_attention(query=texto, key=imagen, value=imagen)`.
4. Calcula la atención cruzada imagen→texto: `self.image_to_text_attention(query=imagen, key=texto, value=texto)`.
5. Quita la dimensión de secuencia (`.squeeze(1)`) y concatena ambos resultados (`dim=1`).
6. Pasa el resultado por `self.classifier` y retorna los logits.

In [ ]:
class CrossModalFusionClassifier(nn.Module):
    def __init__(self, text_encoder, image_encoder, num_classes, freeze_text=False, freeze_image=False,
                 text_dim=TEXT_EMB_DIM, image_dim=IMAGE_EMB_DIM, hidden_dim=256, dropout=0.3, num_heads=2):
        super().__init__()
        self.text_encoder = text_encoder
        self.image_encoder = image_encoder

        for param in self.text_encoder.parameters():
            param.requires_grad = not freeze_text
        for param in self.image_encoder.parameters():
            param.requires_grad = not freeze_image

        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.image_proj = nn.Linear(image_dim, hidden_dim)

        self.text_to_image_attention = MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)
        self.image_to_text_attention = MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, attention_mask, image):
        # TODO: implementa la fusión con atención cruzada (cross-modal attention)
        # 1. text_embedding = ...  / image_embedding = ...  (igual que en NaiveFusionClassifier)
        # 2. text_h = self.text_proj(text_embedding).unsqueeze(1)
        #    image_h = self.image_proj(image_embedding).unsqueeze(1)
        # 3. att_text, _ = self.text_to_image_attention(text_h, image_h, image_h)
        # 4. att_image, _ = self.image_to_text_attention(image_h, text_h, text_h)
        # 5. combined = torch.cat((att_text.squeeze(1), att_image.squeeze(1)), dim=1)
        # 6. return self.classifier(combined)
        raise NotImplementedError("Completa el forward de CrossModalFusionClassifier")

## 📝 Ejecuta el Experimento 3 (Cross-Modal Fusion)

Igual que antes: (a) crea encoders frescos con `build_encoders()`, (b) instancia `CrossModalFusionClassifier` con `freeze_text=False, freeze_image=False`, (c) llama a `train_model(...)`, y (d) guarda los históricos en `exp3_train_losses, exp3_dev_losses, exp3_train_accs, exp3_dev_accs`.

In [ ]:
# TODO: Experimento 3 — Cross-Modal Fusion con RoBERTa y EfficientNet TODO ENTRENABLE
# text_encoder, image_encoder = build_encoders()
# model_exp3 = CrossModalFusionClassifier(text_encoder, image_encoder, num_classes=num_classes,
#                                          freeze_text=False, freeze_image=False)
# exp3_train_losses, exp3_dev_losses, exp3_train_accs, exp3_dev_accs = train_model(
#     model_exp3, train_loader, dev_loader, num_epochs=3, lr=1e-5
# )
# print_plots_losses(exp3_train_losses, exp3_dev_losses)

In [ ]:
# Evaluación final en DEV y TEST (requiere haber corrido el Ejercicio 3)
dev_acc_exp1, dev_acc_exp2, dev_acc_exp3 = exp1_dev_accs[-1], exp2_dev_accs[-1], exp3_dev_accs[-1]
test_acc_exp1, test_f1_exp1 = evaluate_model(model_exp1, test_loader)
test_acc_exp2, test_f1_exp2 = evaluate_model(model_exp2, test_loader)
test_acc_exp3, test_f1_exp3 = evaluate_model(model_exp3, test_loader)

print(f"Naive (frozen):          dev_acc={dev_acc_exp1:.3f}  test_acc={test_acc_exp1:.3f}  test_f1={test_f1_exp1:.3f}")
print(f"Naive (trainable):       dev_acc={dev_acc_exp2:.3f}  test_acc={test_acc_exp2:.3f}  test_f1={test_f1_exp2:.3f}")
print(f"Cross-modal (trainable): dev_acc={dev_acc_exp3:.3f}  test_acc={test_acc_exp3:.3f}  test_f1={test_f1_exp3:.3f}")

labels = ["Naive (frozen)", "Naive (trainable)", "Cross-modal (trainable)"]
dev_accs_final = [dev_acc_exp1, dev_acc_exp2, dev_acc_exp3]
test_accs_final = [test_acc_exp1, test_acc_exp2, test_acc_exp3]

x = np.arange(len(labels))
width = 0.35
plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, dev_accs_final, width, label="Dev")
plt.bar(x + width / 2, test_accs_final, width, label="Test")
plt.ylabel("Accuracy")
plt.title("Comparación de los 3 experimentos (dev vs. test)")
plt.xticks(x, labels, rotation=15)
plt.legend()
plt.show()

print({"dev": dict(zip(labels, dev_accs_final)), "test": dict(zip(labels, test_accs_final))})

## Preguntas

* ¿Cuál es el impacto de descongelar los pesos de los encoders (Experimento 1 vs 2)?
* ¿La fusión con atención cruzada (Experimento 3) mejora respecto a la fusión naive (Experimento 2)?
* ¿Qué costo (tiempo, memoria) tiene cada experimento?

### Respuestas

**Ponga aca tus respuestas**

---
## Parte 2: Modelo instruido en modo zero-shot

En esta parte usaremos un modelo pre entrenado de tipo *instruct*, es decir, un modelo que recibe instrucciones y puede ser utilizado sin realizar ningún fine-tuning. Lo usaremos en modalidad **zero-shot**, ver [zero-shot learning](https://huggingface.co/tasks/zero-shot-classification).

Algunos modelos instruidos multimodales disponibles en Hugging Face:

   * **[Gemma3 (multimodal, multilingual, long context open LLM)](https://huggingface.co/blog/gemma3)** - `'google/gemma-3-4b-it'`
   * **[Qwen3-VL](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct)** - `'Qwen/Qwen3-VL-2B-Instruct'`
   * **[BLIP3](https://huggingface.co/Salesforce/xgen-mm-phi3-mini-instruct-r-v1)** - `'Salesforce/xgen-mm-phi3-mini-instruct-r-v1'`

En este lab usaremos **`Qwen/Qwen3-VL-2B-Instruct`** para clasificar directamente (sin entrenar nada) algunos tweets del conjunto de test en una de las 5 categorías del dataset.

In [ ]:
from transformers import pipeline

# Las clases vienen directamente del dataset (mismas categorías usadas por LabelEncoder en la Parte 1)
CLASSES = list(le.classes_)
print("Clases:", CLASSES)

vlm_pipeline = pipeline(
    task="image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=torch.bfloat16,
)

# Ejemplo inicial: el clásico ejemplo de la "tortuga en el dulce" (candy) de Lab4_solution (con Gemma3)
import requests
from io import BytesIO

candy_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
candy_question = "What animal is on the candy?"

candy_messages = [
    {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant."}]},
    {"role": "user", "content": [
        {"type": "image", "url": candy_url},
        {"type": "text", "text": candy_question},
    ]},
]
candy_output = vlm_pipeline(text=candy_messages, max_new_tokens=30)
candy_answer = candy_output[0]["generated_text"][-1]["content"]

candy_img = Image.open(BytesIO(requests.get(candy_url).content)).convert("RGB")
plt.imshow(candy_img)
plt.axis("off")
plt.show()

print(f"Pregunta: {candy_question}")
print(f"Respuesta del modelo: {candy_answer}")

El modelo te parece bien? 

In [ ]:
def classify_zero_shot(tweet_text, image_path, classes=CLASSES):
    prompt = (
        f"You will classify a tweet (with its image) about a crisis/disaster into exactly one "
        f"of these categories: {', '.join(classes)}.\n"
        f"Tweet: \"{tweet_text}\"\n"
        f"Answer with ONLY the category name, nothing else."
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant that classifies crisis-related tweets."}]},
        {"role": "user", "content": [
            {"type": "image", "url": image_path},
            {"type": "text", "text": prompt},
        ]},
    ]
    output = vlm_pipeline(text=messages, max_new_tokens=20)
    return output[0]["generated_text"][-1]["content"].strip()


# Ejemplo simple: probamos el pipeline en un solo tweet de test
# (mismo estilo que el ejemplo de la "tortuga" del Lab4_solution con Gemma3)
example_idx = 0
example_tweet = X_test.iloc[example_idx]
example_true_label = y_test.iloc[example_idx]

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant that classifies crisis-related tweets."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "url": path_image_test[example_idx]},
            {"type": "text", "text": (
                f"You will classify a tweet (with its image) about a crisis/disaster into exactly one "
                f"of these categories: {', '.join(CLASSES)}.\n"
                f"Tweet: \"{example_tweet}\"\n"
                f"Answer with ONLY the category name, nothing else."
            )}
        ]
    }
]

output = vlm_pipeline(text=messages, max_new_tokens=20)
example_pred = output[0]["generated_text"][-1]["content"].strip()

img = Image.open(path_image_test[example_idx]).convert("RGB")
plt.imshow(img)
plt.axis("off")
plt.title(f"Predicción: {example_pred}  |  Real: {example_true_label}")
plt.show()

print(f"Pregunta (tweet): {example_tweet}")
print(f"Respuesta del modelo (Qwen3-VL): {example_pred}")
print(f"Etiqueta real:                   {example_true_label}")

In [ ]:
# Ahora evaluamos sobre varios ejemplos del test set (evaluación aproximada, no exhaustiva)
N_ZERO_SHOT_EXAMPLES = 10
sample_idx = np.random.choice(len(X_test), N_ZERO_SHOT_EXAMPLES, replace=False)

correct = 0
for idx in sample_idx:
    pred = classify_zero_shot(X_test.iloc[idx], path_image_test[idx])
    true_label = y_test.iloc[idx]
    match = pred.strip().lower() == true_label.strip().lower()
    correct += int(match)
    print(f"Tweet: {X_test.iloc[idx][:80]}...")
    print(f"  Predicción Qwen3-VL: {pred}")
    print(f"  Etiqueta real:       {true_label}")
    print(f"  {'✅ Coincide' if match else '❌ No coincide'}\n")

print(f"Accuracy aproximada (zero-shot, {N_ZERO_SHOT_EXAMPLES} ejemplos): {correct / N_ZERO_SHOT_EXAMPLES:.0%}")

## Preguntas finales

* ¿Cómo se compara el desempeño zero-shot de Qwen3-VL con los modelos entrenados en la Parte 1?
* ¿Cuál es el interés de los modelos de tipo Instruct? ¿Qué observaste a partir de los resultados?

### Respuesta

**Ponga aca tus respuestas**